<a href="https://colab.research.google.com/github/soumaya-lamrhari/customer-conversion-prediction/blob/main/notebooks/4_model_explainability.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Model Explainability

## Objective

After evaluating the XGBoost model, the next step is to understand what is driving its predictions.

In this notebook, I use SHAP to identify the features that have the strongest influence on conversion predictions and understand how they affect the model’s output.

## Setup

In [1]:
!pip install -q ucimlrepo xgboost shap

import pandas as pd
import numpy as np

from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier


# Load the Bank Marketing dataset
bank_marketing = fetch_ucirepo(id=222)

X = bank_marketing.data.features.copy()
y = bank_marketing.data.targets.copy()

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (45211, 16)
Target shape: (45211, 1)


## Data Preparation

In [2]:
# Apply the same preprocessing decisions used during model development
X = X.drop(columns=["duration"])

X["poutcome"] = X["poutcome"].fillna("no_previous_contact")
X["contact"] = X["contact"].fillna("unknown")
X["job"] = X["job"].fillna("unknown")
X["education"] = X["education"].fillna("unknown")

X = X.rename(columns={"day_of_week": "day_of_month"})

# Convert the target to binary values
y = y["y"].map({"yes": 1, "no": 0})

print("Features shape:", X.shape)
print("Missing values:", X.isna().sum().sum())
print("Conversion rate:", round(y.mean() * 100, 2), "%")

Features shape: (45211, 15)
Missing values: 0
Conversion rate: 11.7 %


In [3]:
# Use the same split as the modeling notebook
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

Training set: (36168, 15)
Test set: (9043, 15)


## Final XGBoost Model

I recreate the final XGBoost model using the same preprocessing steps and parameters from the modeling notebook. The model is trained on the full training set so I can use SHAP to understand what is driving its predictions.

In [4]:
# Identify numerical and categorical features
numerical_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()

categorical_features = X_train.select_dtypes(include=["object"]).columns.tolist()

print("Numerical features:", numerical_features)
print("Categorical features:", categorical_features)

Numerical features: ['age', 'balance', 'day_of_month', 'campaign', 'pdays', 'previous']
Categorical features: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']


In [5]:
# Encode categorical features and keep numerical features unchanged
xgb_preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", "passthrough", numerical_features)
    ]
)

In [6]:
# Create the final XGBoost pipeline
xgb_model = Pipeline(
    steps=[
        ("preprocessor", xgb_preprocessor),
        ("classifier", XGBClassifier(
            n_estimators=200,
            max_depth=4,
            learning_rate=0.05,
            random_state=42,
            eval_metric="logloss"
        ))
    ]
)

In [7]:
# Train the final model on the full training set
xgb_model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['job', 'marital',
                                                   'education', 'default',
                                                   'housing', 'loan', 'contact',
                                                   'month', 'poutcome']),
                                                 ('num', 'passthrough',
                                                  ['age', 'balance',
                                                   'day_of_month', 'campaign',
                                                   'pdays', 'previous'])])),
                ('classifier',
                 XGBClassifier(base_score=None, booster=None, callback...
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.05,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=4, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=200, n_jobs=None,
                               num_parallel_tree=None, ...))])

## SHAP Analysis

I use SHAP to understand which features have the strongest influence on the model’s conversion predictions and how they affect those predictions.

In [8]:
# Get the trained preprocessing and XGBoost components
trained_preprocessor = xgb_model.named_steps["preprocessor"]
trained_xgb = xgb_model.named_steps["classifier"]

# Transform the test data using the fitted preprocessor
X_test_transformed = trained_preprocessor.transform(X_test)

# Get the feature names after one-hot encoding so we can interpret the SHAP results
feature_names = trained_preprocessor.get_feature_names_out()

print("Transformed test shape:", X_test_transformed.shape)
print("Number of feature names:", len(feature_names))

Transformed test shape: (9043, 50)
Number of feature names: 50
